# 05 — Hyperparameter Tuning (KAGGLE GPU)
## WavSent-MTL · Tasks 2.1–2.9

**Runs on: Kaggle T4 x2 GPU**

**What this notebook does:**
- Random search: 40 trials per model (TKAN, LSTM, GRU, TCN)
- Primary search metric: val_loss (composite optimization — correct for hyperparameter selection)
- val_binary_accuracy is tracked and logged alongside val_loss in every trial
- Done ONCE — same best params applied to Kaggle dataset
- Saves: `best_params_{model}.json` for each model

**Training configuration (from config.py):**
- max_epochs: 150 (full training runs — not search trials)
- early_stopping_patience: 35 (monitor=val_binary_accuracy)
- lr_reduce_patience: 10 (monitor=val_loss)
- Search trials use a short 50-epoch cap with patience=10 on val_loss

**After this notebook:**
1. Download all 4 `best_params_*.json` files
2. Manually update `config/config.py` `best_params` section
3. Push updated config to GitHub

In [ ]:
# ── Kaggle Setup ────────────────────────────────────────────────────────────
import subprocess
subprocess.run(['git', 'clone', 'https://github.com/IAMNEERAJ05/WavSent-MTL.git',
                '/kaggle/working/WavSent-MTL'], check=True)

import sys
import os
sys.path.insert(0, '/kaggle/working/WavSent-MTL')
os.chdir('/kaggle/working/WavSent-MTL')

import torch
from config.config import CONFIG

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}')
print(f'Device:  {device}')
if device == 'cuda':
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
print('CONFIG loaded.')

## Load Kotekar Processed Arrays

In [ ]:
import numpy as np
import json

# Kaggle dataset path — upload data/processed/kotekar/ as 'wavsent-kotekar-processed'
KOTEKAR_INPUT = '/kaggle/input/wavsent-kotekar-processed'

data = {
    'X_train':      np.load(f'{KOTEKAR_INPUT}/X_train.npy'),
    'X_val':        np.load(f'{KOTEKAR_INPUT}/X_val.npy'),
    'X_test':       np.load(f'{KOTEKAR_INPUT}/X_test.npy'),
    'y_clf_train':  np.load(f'{KOTEKAR_INPUT}/y_clf_train.npy'),
    'y_clf_val':    np.load(f'{KOTEKAR_INPUT}/y_clf_val.npy'),
    'y_clf_test':   np.load(f'{KOTEKAR_INPUT}/y_clf_test.npy'),
    'y_reg_train':  np.load(f'{KOTEKAR_INPUT}/y_reg_train.npy'),
    'y_reg_val':    np.load(f'{KOTEKAR_INPUT}/y_reg_val.npy'),
    'y_reg_test':   np.load(f'{KOTEKAR_INPUT}/y_reg_test.npy'),
}

with open(f'{KOTEKAR_INPUT}/selected_features.json') as f:
    selected_features = json.load(f)

N_FEATURES = data['X_train'].shape[2]   # 8
print(f'X_train:          {data["X_train"].shape}')
print(f'X_val:            {data["X_val"].shape}')
print(f'n_features:       {N_FEATURES}')
print(f'selected_features:{selected_features}')

## Tasks 2.3–2.6 — Random Search (40 trials × 4 models)

> Each model takes ~5-10 min on T4 GPU. Total ~30-40 min.

In [ ]:
from src.training.hyperparam_tuning import random_search

best_params_all = {}

for model_name in CONFIG['pso_models']:   # ['tkan', 'lstm', 'gru', 'tcn']
    print(f'\n{"=" * 60}')
    print(f'Random search: {model_name.upper()}  ({CONFIG["n_search_trials"]} trials)')
    print(f'{"=" * 60}')

    best = random_search(
        model_name=model_name,
        n_features=N_FEATURES,
        data=data,
        n_trials=CONFIG['n_search_trials'],
        device=device,
    )

    best_params_all[model_name] = best

    # Save immediately after each model
    out_path = f'/kaggle/working/best_params_{model_name}.json'
    with open(out_path, 'w') as f:
        json.dump(best, f, indent=2)
    print(f'Saved: {out_path}')

    # Clear GPU memory between models
    import gc
    torch.cuda.empty_cache()
    gc.collect()

## Summary of Best Params

In [ ]:
import pandas as pd

print('=' * 60)
print('Best hyperparameters per model:')
print('=' * 60)
for model_name, params in best_params_all.items():
    print(f'\n{model_name.upper()}:')
    for k, v in params.items():
        print(f'  {k}: {v}')

print()
print('Search metric: val_loss (primary — composite optimization)')
print('val_binary_accuracy was tracked alongside val_loss each trial.')
print()
print('Full training runs will use (from config.py):')
print(f"  early_stopping_patience : {CONFIG['early_stopping_patience']}")
print(f"  early_stopping_monitor  : {CONFIG['early_stopping_monitor']}")
print(f"  max_epochs              : {CONFIG['max_epochs']}")
print(f"  lr_reduce_patience      : {CONFIG['lr_reduce_patience']}")
print(f"  lr_reduce_monitor       : {CONFIG['lr_reduce_monitor']}")
print()
print('Files saved to /kaggle/working/:')
for model_name in best_params_all:
    print(f'  best_params_{model_name}.json')

print()
print('NEXT STEPS:')
print('1. Download all 4 best_params_*.json files')
print('2. Update config/config.py best_params section manually')
print('3. Push config to GitHub')
print('4. Run notebook 06_training_kotekar')